# Error Analysis of Tr-PINNs Algorithm for 2D Incompressible Navier-Stokes Equations with Non-Homogeneous Boundary Conditions

**Paper:** Liu, D., Li, X., Yang, R. (2026). *Error Analysis of Tr-PINNs Algorithm for 2D Incompressible Navier-Stokes Equations with Non-Homogeneous Boundary Conditions.* arXiv:2606.06268 [math.NA].

**Carpeta origen:** `PINNs/1. mecanica de fluidos/Error Analysis of Tr-PINNs Algorithm for 2D Incompressible.pdf`

## Como se usan las PINNs en este paper

El paper resuelve las ecuaciones de Navier-Stokes 2D incompresibles no estacionarias con condiciones de contorno **no homogeneas** (Eq. 1.1):

$$\partial_t u - \nu\Delta u + u\cdot\nabla u + \nabla p = f,\quad \nabla\cdot u=0,\quad u|_{t=0}=u_0,\quad u|_{\partial\Omega}=g$$

Una PINN convencional mide el error de contorno solo con la norma $L^2$. El paper muestra que esto es insuficiente: si la solucion real solo pertenece a $H^1$ (no $H^2$, como ocurre con fronteras irregulares, discontinuidades de material, etc.), el error de contorno en $L^2$ no controla el error global. Proponen **Tr-PINNs**, que an~ade al residuo de contorno un termino extra basado en el **teorema de la traza**, midiendo el residuo con la (semi)norma $H^{1/2}$ en vez de $L^2$ (Eq. 1.5). En la practica (Eq. 1.6), esta seminorma se aproxima con un termino tipo diferencias finitas entre puntos de frontera vecinos:

$$\varepsilon_T^{b_2}(h,S_{b_2}) = \frac{|\partial\Omega|^2 T}{N_b\,k}\sum_{i=1}^{N_b}\sum_{j\in\mathcal{N}_k(i)} \frac{|\mathcal{R}_{bdy}(t^b_i,x^b_i)-\mathcal{R}_{bdy}(t^b_j,x^b_j)|^2}{|x_i-x_j|^2}$$

es decir, penaliza que el **residuo de contorno varie de forma brusca entre puntos de frontera cercanos** ($\mathcal{N}_k(i)$ = k vecinos mas cercanos de $i$), ademas del termino $L^2$ estandar $\varepsilon_T^{b_1}$. El paper demuestra (Teorema 1.1) que el error de la solucion PINN esta acotado por este error de generalizacion reforzado, y valida numericamente con vortices de Taylor-Green que Tr-PINNs mejora sustancialmente la precision frente a la PINN convencional.

Este cuaderno reproduce fielmente: la arquitectura PINN estandar para Navier-Stokes 2D no estacionario, el benchmark de Taylor-Green (solucion analitica exacta con decaimiento viscoso), y compara **PINN convencional (solo $L^2$ en el contorno)** vs. **Tr-PINNs ($L^2$ + termino de traza $H^{1/2}$ discreto, Eq. 1.6)**.

## Repositorio publico de referencia

El PDF no incluye un repositorio propio para Tr-PINNs. Como referencia publica general de PINNs aplicadas a Navier-Stokes (el mismo tipo de problema base sobre el que se construye Tr-PINNs), se usa:

- **mattialoszach/navier-stokes-pinn** &mdash; https://github.com/mattialoszach/navier-stokes-pinn — PINN para las ecuaciones de Navier-Stokes en PyTorch.

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Benchmark de Taylor-Green (solucion exacta usada para validacion, Seccion 4)

Dominio $\Omega=[0,2\pi]^2$, $t\in[0,T]$. El vortice de Taylor-Green decaindo viscosamente resuelve exactamente Navier-Stokes:

$$u=\cos(x)\sin(y)e^{-2\nu t},\quad v=-\sin(x)\cos(y)e^{-2\nu t},\quad p=-\tfrac{1}{4}(\cos 2x+\cos 2y)e^{-4\nu t}$$

In [ ]:
nu = 0.1
T_max = 1.0
L = 2 * np.pi

def exact_solution(t, x, y):
    decay1 = torch.exp(-2 * nu * t)
    decay2 = torch.exp(-4 * nu * t)
    u = torch.cos(x) * torch.sin(y) * decay1
    v = -torch.sin(x) * torch.cos(y) * decay1
    p = -0.25 * (torch.cos(2 * x) + torch.cos(2 * y)) * decay2
    return u, v, p

## 2. Red PINN para (t,x,y) -> (u,v,p)

In [ ]:
class PINN(nn.Module):
    def __init__(self, n_hidden=4, n_neurons=40):
        super().__init__()
        layers = [nn.Linear(3, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 3)]
        self.net = nn.Sequential(*layers)

    def forward(self, txy):
        out = self.net(txy)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3]


def d_d(f, v):
    return torch.autograd.grad(f, v, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]

## 3. Puntos de colocacion: interior, condicion inicial y frontera (con vecinos para el termino de traza, Eq. 1.6-1.7)

In [ ]:
def rand_txy(n, t_range=(0, T_max)):
    t = torch.rand(n, 1) * (t_range[1] - t_range[0]) + t_range[0]
    x = torch.rand(n, 1) * L
    y = torch.rand(n, 1) * L
    return torch.cat([t, x, y], dim=1).to(device).requires_grad_(True)

N_i, N_init, N_b = 800, 200, 200
txy_interior = rand_txy(N_i)

txy_init = rand_txy(N_init, t_range=(0, 0))

# Puntos de frontera: se muestrea un parametro de arco 's' a lo largo del perimetro
# del cuadrado [0,L]x[0,L], para poder definir vecinos consecutivos (Eq. 1.6, N_k(i)).
s = torch.rand(N_b) * (4 * L)
bx = torch.zeros(N_b)
by = torch.zeros(N_b)
seg = (s // L).long().clamp(max=3)
frac = s - seg.float() * L
for i in range(N_b):
    if seg[i] == 0:
        bx[i], by[i] = frac[i], 0.0
    elif seg[i] == 1:
        bx[i], by[i] = L, frac[i]
    elif seg[i] == 2:
        bx[i], by[i] = L - frac[i], L
    else:
        bx[i], by[i] = 0.0, L - frac[i]
order = torch.argsort(s)
s_sorted, bx, by = s[order], bx[order], by[order]
t_b = torch.rand(N_b) * T_max
txy_bdy = torch.stack([t_b, bx, by], dim=1).to(device).requires_grad_(True)
arc_s = s_sorted.to(device)

## 4. Funcion de perdida: PINN convencional (solo $L^2$ en frontera) vs. Tr-PINNs ($L^2$ + traza $H^{1/2}$, Eq. 1.6)

In [ ]:
def pde_residual(model, txy):
    u, v, p = model(txy)
    grads_u = d_d(u, txy); u_t, u_x, u_y = grads_u[:, 0:1], grads_u[:, 1:2], grads_u[:, 2:3]
    grads_v = d_d(v, txy); v_t, v_x, v_y = grads_v[:, 0:1], grads_v[:, 1:2], grads_v[:, 2:3]
    grads_p = d_d(p, txy); p_x, p_y = grads_p[:, 1:2], grads_p[:, 2:3]
    u_xx = d_d(u_x, txy)[:, 1:2]; u_yy = d_d(u_y, txy)[:, 2:3]
    v_xx = d_d(v_x, txy)[:, 1:2]; v_yy = d_d(v_y, txy)[:, 2:3]

    res_u = u_t - nu * (u_xx + u_yy) + u * u_x + v * u_y + p_x
    res_v = v_t - nu * (v_xx + v_yy) + u * v_x + v * v_y + p_y
    res_div = u_x + v_y
    return res_u, res_v, res_div


def boundary_residual(model, txy):
    u, v, _ = model(txy)
    u_e, v_e, _ = exact_solution(txy[:, 0:1], txy[:, 1:2], txy[:, 2:3])
    return u - u_e, v - v_e


def trace_term(res_u, res_v, arc_s, k=5):
    """Aproximacion discreta de la seminorma H^{1/2} de contorno (Eq. 1.6, eps_T^{b2}):
    penaliza la variacion del residuo de frontera entre sus k vecinos mas cercanos en arco."""
    n = res_u.shape[0]
    res = torch.cat([res_u, res_v], dim=1)  # (n, 2)
    total = 0.0
    count = 0
    for shift in list(range(1, k // 2 + 1)) + list(range(-(k // 2), 0)):
        idx = (torch.arange(n, device=res.device) + shift) % n
        d_res2 = torch.sum((res - res[idx])**2, dim=1)
        d_s2 = (arc_s - arc_s[idx])**2 + 1e-6
        total = total + torch.sum(d_res2 / d_s2)
        count += n
    return total / count


def compute_loss(model, use_trace_term):
    res_u, res_v, res_div = pde_residual(model, txy_interior)
    loss_pde = torch.mean(res_u**2) + torch.mean(res_v**2) + torch.mean(res_div**2)

    u0, v0, _ = model(txy_init)
    u0_e, v0_e, _ = exact_solution(txy_init[:, 0:1], txy_init[:, 1:2], txy_init[:, 2:3])
    loss_init = torch.mean((u0 - u0_e)**2 + (v0 - v0_e)**2)

    res_u_b, res_v_b = boundary_residual(model, txy_bdy)
    loss_bdy_l2 = torch.mean(res_u_b**2 + res_v_b**2)

    loss = loss_pde + loss_init + loss_bdy_l2
    if use_trace_term:
        loss = loss + trace_term(res_u_b, res_v_b, arc_s)
    return loss

## 5. Entrenamiento: PINN convencional vs Tr-PINNs

In [ ]:
def train(use_trace_term, epochs=3000, lr=1e-3):
    model = PINN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        opt.zero_grad()
        loss = compute_loss(model, use_trace_term)
        loss.backward()
        opt.step()
        history.append(loss.item())
        if epoch % 500 == 0:
            print(f'[{"Tr-PINNs" if use_trace_term else "PINN conv."}] epoch {epoch:5d} | loss={loss.item():.4e}')
    return model, history


model_conv, hist_conv = train(use_trace_term=False)
model_tr, hist_tr = train(use_trace_term=True)

## 6. Evaluacion: error $L^2$ contra la solucion exacta de Taylor-Green (reproduce el hallazgo de la Seccion 4)

In [ ]:
def l2_error(model, n_test=4000):
    txy = rand_txy(n_test)
    with torch.no_grad():
        u_p, v_p, _ = model(txy)
        u_e, v_e, _ = exact_solution(txy[:, 0:1], txy[:, 1:2], txy[:, 2:3])
        err = torch.sqrt(torch.mean((u_p - u_e)**2 + (v_p - v_e)**2))
    return err.item()


err_conv = l2_error(model_conv)
err_tr = l2_error(model_tr)
print(f'Error L2 (velocidad) - PINN convencional: {err_conv:.4f}')
print(f'Error L2 (velocidad) - Tr-PINNs         : {err_tr:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(['PINN convencional', 'Tr-PINNs'], [err_conv, err_tr], color=['tab:orange', 'tab:blue'])
axes[0].set_ylabel('Error $L^2$ de velocidad')
axes[0].set_title('Precision: PINN convencional vs Tr-PINNs')

axes[1].semilogy(hist_conv, label='PINN convencional')
axes[1].semilogy(hist_tr, label='Tr-PINNs')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss (escala log)')
axes[1].set_title('Convergencia del entrenamiento')
axes[1].legend()
plt.tight_layout()
plt.show()

Se espera (como en el paper) que **Tr-PINNs logre un error menor** que la PINN convencional gracias al termino adicional de traza $H^{1/2}$ (Eq. 1.6), que fuerza al residuo de frontera a variar suavemente entre puntos vecinos en lugar de solo minimizar su magnitud puntual. El Teorema 1.1 del paper garantiza que esta mejora en el error de generalizacion de frontera se traduce en una cota de error global mas ajustada.